# 01 – Exploring the PettingZoo Connect Four environment

Goal: understand how the game is represented before we build anything on top of it.

Questions this notebook answers:
1. What does an observation look like, and from whose perspective is it?
2. How are legal moves communicated (action mask)?
3. How do turns and rewards work in PettingZoo's AEC (turn-based) API?

In [1]:
import numpy as np
from pettingzoo.classic import connect_four_v3

env = connect_four_v3.env()
env.reset(seed=0)

print("Agents:           ", env.agents)
print("Action space:     ", env.action_space("player_0"))
print("Observation space:", env.observation_space("player_0"))

Agents:            ['player_0', 'player_1']
Action space:      Discrete(7)
Observation space: Dict('action_mask': Box(0, 1, (7,), int8), 'observation': Box(0, 1, (6, 7, 2), int8))


## 1. The observation

`env.last()` returns the observation **for the agent whose turn it is** (`env.agent_selection`),
plus its reward, termination/truncation flags and an info dict.

The observation is a dict:
- `observation`: shape `(6, 7, 2)` → 6 rows × 7 columns × 2 planes. Row 0 is the **top** of the board.
- `action_mask`: length 7, 1 = column is legal.

In [2]:
obs, reward, termination, truncation, info = env.last()
print("To move:     ", env.agent_selection)
print("Obs shape:   ", obs["observation"].shape, obs["observation"].dtype)
print("Action mask: ", obs["action_mask"])

To move:      player_0
Obs shape:    (6, 7, 2) int8
Action mask:  [1 1 1 1 1 1 1]


A small helper to print the board. We draw it from the **current player's perspective**:
`X` = pieces in plane 0 (the player to move), `O` = pieces in plane 1 (the opponent).

In [3]:
def show(obs):
    board = obs["observation"]
    symbols = np.full((6, 7), ".")
    symbols[board[:, :, 0] == 1] = "X"   # player to move
    symbols[board[:, :, 1] == 1] = "O"   # opponent
    print("\n".join(" ".join(row) for row in symbols))
    print("0 1 2 3 4 5 6   <- columns")
    print("mask:", obs["action_mask"])

show(obs)

. . . . . . .
. . . . . . .
. . . . . . .
. . . . . . .
. . . . . . .
. . . . . . .
0 1 2 3 4 5 6   <- columns
mask: [1 1 1 1 1 1 1]


## 2. The perspective flips every move

Let `player_0` play column 3, then look at the board from `player_1`'s side, and so on.
Watch how the **same physical piece** switches between `X` and `O`.

In [4]:
env.reset(seed=0)
for col in [3, 3, 4]:
    mover = env.agent_selection
    env.step(col)
    obs, *_ = env.last()
    print(f"{mover} played column {col}. Now {env.agent_selection} to move, sees:")
    show(obs)
    print()

player_0 played column 3. Now player_1 to move, sees:
. . . . . . .
. . . . . . .
. . . . . . .
. . . . . . .
. . . . . . .
. . . O . . .
0 1 2 3 4 5 6   <- columns
mask: [1 1 1 1 1 1 1]

player_1 played column 3. Now player_0 to move, sees:
. . . . . . .
. . . . . . .
. . . . . . .
. . . . . . .
. . . O . . .
. . . X . . .
0 1 2 3 4 5 6   <- columns
mask: [1 1 1 1 1 1 1]

player_0 played column 4. Now player_1 to move, sees:
. . . . . . .
. . . . . . .
. . . . . . .
. . . . . . .
. . . X . . .
. . . O O . .
0 1 2 3 4 5 6   <- columns
mask: [1 1 1 1 1 1 1]



**Why this matters:** the network will always see "my pieces" vs "opponent's pieces",
so it never needs to know whether it is player 0 or 1. One network can play both sides.
The flip side: the value it predicts is also "good for *me*", so values must change sign
when we go from one player's turn to the other's (see `RL_Background.md`, Section 3).

For comparison, the raw internal board (`env.unwrapped.board`) is a flat list of 42 cells with
fixed player IDs (1 = player_0, 2 = player_1), independent of who is to move:

In [5]:
print(np.array(env.unwrapped.board).reshape(6, 7))

[[0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0]
 [0 0 0 2 0 0 0]
 [0 0 0 1 1 0 0]]


## 3. The action mask

Fill column 0 completely (6 pieces) and check that it becomes illegal.

In [6]:
env.reset(seed=0)
for _ in range(6):
    env.step(0)
obs, *_ = env.last()
show(obs)

O . . . . . .
X . . . . . .
O . . . . . .
X . . . . . .
O . . . . . .
X . . . . . .
0 1 2 3 4 5 6   <- columns
mask: [0 1 1 1 1 1 1]


## 4. A full game with random moves

The standard AEC loop: iterate over agents; each one reads `env.last()` and acts.
Once the game is over, every agent still gets one more turn where it must pass `None`
(this is how PettingZoo delivers the final reward to each agent).

In [7]:
rng = np.random.default_rng(0)

def random_move(obs):
    legal = np.flatnonzero(obs["action_mask"])
    return int(rng.choice(legal))

env.reset(seed=0)
n_moves = 0
for agent in env.agent_iter():
    obs, reward, termination, truncation, info = env.last()
    if termination or truncation:
        print(f"{agent}: final reward {reward}")
        action = None
    else:
        action = random_move(obs)
        n_moves += 1
    env.step(action)

print("Moves played:", n_moves)
print(np.array(env.unwrapped.board).reshape(6, 7))

player_0: final reward -1
player_1: final reward 1
Moves played: 20
[[0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0]
 [0 0 0 1 1 0 0]
 [2 0 0 2 2 2 2]
 [1 1 0 1 1 2 1]
 [2 2 1 1 2 1 2]]


## 5. A forced win: checking the reward signal

`player_0` stacks column 0, `player_1` stacks column 1 → `player_0` wins with a vertical four
on its 4th move. Rewards: +1 for the winner, −1 for the loser, and **0 on every non-final step**
(the sparse reward problem from the background file).

In [8]:
env.reset(seed=0)
moves = iter([0, 1, 0, 1, 0, 1, 0])
for agent in env.agent_iter():
    obs, reward, termination, truncation, info = env.last()
    action = None if termination or truncation else next(moves)
    print(f"{agent:9s} reward={reward:+d}  done={termination}  action={action}")
    env.step(action)

player_0  reward=+0  done=False  action=0
player_1  reward=+0  done=False  action=1
player_0  reward=+0  done=False  action=0
player_1  reward=+0  done=False  action=1
player_0  reward=+0  done=False  action=0
player_1  reward=+0  done=False  action=1
player_0  reward=+0  done=False  action=0
player_1  reward=-1  done=True  action=None
player_0  reward=+1  done=True  action=None


## 6. Random vs random statistics

How often does the first player win when both play randomly? This is our first data point about
the first-player advantage, and a reference for later baselines.

In [12]:
results = []
for g in range(1000):
    env.reset(seed=g)
    final = {}
    for agent in env.agent_iter():
        obs, reward, termination, truncation, info = env.last()
        if termination or truncation:
            final[agent] = reward
            env.step(None)
        else:
            env.step(random_move(obs))
    results.append(final["player_0"])

results = np.array(results)
print(f"player_0 wins: {np.mean(results == 1):.1%}")
print(f"player_1 wins: {np.mean(results == -1):.1%}")
print(f"draws:         {np.mean(results == 0):.1%}")

player_0 wins: 53.6%
player_1 wins: 46.1%
draws:         0.3%


## Things to try

- How long (in moves) are random games on average?
- Render a game visually: `connect_four_v3.env(render_mode="human")` opens a pygame window.
- Copying the environment: time `copy.deepcopy(env)` vs. copying a NumPy array of shape `(6, 7)`.
  MCTS needs to copy the game state *thousands* of times per move. This is why Step 3 builds
  our own lightweight game class.